# Basic API

The code below shows how to use Knwler as an API. You can take out the pipeline parts you need in your context.
Note that the methods shown below are wrappers in order to make it as easy as possible to perform various tasks via a single namespace, i.e. the `knwler.api` import. You can import the underlying methods as well but are usually somewhat more involved.

This notebook is using Knwler v1.0.6 or above.



## Install Knwler

There are [various ways you can install knwler](https://knwler.com/docs/setup.html), the easiest being to use `uv add knwler` after running `uv init` in a directory. You can ensure it's installed via something like:

In [ ]:
import knwler
from importlib.metadata import version
print("knwler v{}".format(version("knwler")))

## Config
The various API methods accept an optional `Config` instance which in essence defines the LLM and its properties. 

In [ ]:
from knwler.api import Config
config = Config(backend="ollama", discovery_model="gemma3:4b", extraction_model="gemma3:12b ")

## Fetching data

The API allows you to fetch an arbitrary url, file or Wikipedia articles.

In [ ]:
from knwler.api import fetch_wikipedia_page
await fetch_wikipedia_page("Python (programming language)")

When fetching a web page or a pdf the fetch method will return a tuple `metadata, content`:

In [ ]:
from knwler.api import fetch_url
import json
metadata, content = await fetch_url("https://www.wikipedia.org/")
print(json.dumps(metadata, indent=2))

In [ ]:
metadata, content = await fetch_url("https://knwler.com/pdfs/HumanRights.pdf")
print(json.dumps(metadata, indent=2))

and the content will be in this case binary. You can save it like so:

In [ ]:
with open("HumanRights.pdf", "wb") as f:
    f.write(content)

## Parsing PDF files

In [ ]:
from knwler.api import parse_pdf
import os

md = await parse_pdf(os.path.join(os.getcwd(), "HumanRights.pdf"))
print(md[:500])

## Chunking

You can use the methods above to fetch markdown or you have some text/markdown available:

In [ ]:
# use some pdf 
md = await parse_pdf(os.path.join(os.getcwd(), "HumanRights.pdf"))

 # or read in text
# with open("~/document.md", "r") as f:
#     text = f.read()
from knwler.api import Config, chunk
config = Config(max_tokens=200, overlap_tokens=20)   
chunks = await chunk(md, config=config)


In [ ]:
print(f"There are {len(chunks)} chunks, the first chunk is:\n\n{chunks[0]}")

## Schema

Simply said, a schema defines the type of words and the relations between these words that matter to you. It is a set of entity types and relationships. You can define them or you can let Knwler decide (via a LLM):

In [ ]:
from knwler.api import infer_schema
from dataclasses import asdict
import json
schema = await infer_schema(md, config=config)
print(json.dumps(asdict(schema), indent=2))

The inferred schema can be changed after discovery, there is nothing holy in what the LLM suggests. If you want to define your own schema you can use something like this:

In [ ]:
from knwler.models import Schema
schema = Schema(entity_types=["person", "size"], relation_types=["has_size"])

## Language

The graph extraction uses internally language detection but if you want to use it standalone you can access it like this:

In [ ]:
from knwler.api import discover_language
dic = {
    "fr": "L'importance de la découverte de la langue ne peut être surestimée.",
    "es": "La importancia del descubrimiento del idioma no puede ser subestimada.",
    "cn":"语言发现的重要性不容小觑。",
    "nl": "Het belang van taalontdekking kan niet worden overschat.",
    "en": "The importance of language discovery cannot be overstated."
    
}
for lang, sentence in dic.items():
    language = await discover_language(sentence, config=config)
    print(f"Detected language for '{sentence}': {language}")    


If you want to set/override the language use the `set_language` method from `knwler.api`.

## Extraction

You can extract a pdf, a piece of text of a collection of chunks with the same API method.

In [ ]:
from knwler.api import extract, Config, Schema
from knwler.language import get_current_language
schema = Schema(entity_types=["animal", "object"], relation_types=["is_on"])
g = await extract("The cat is on the table.", schema, config=Config())

The result contains the schema, the knowledge graph and the chunks:

In [ ]:
import json
from dataclasses import asdict
print(json.dumps(asdict(g), indent=2))

You can also give the method a pdf. In this short pdf there are nine people coming from various countries:

In [ ]:
from knwler.api import Chunk, Schema, Config, extract
from pathlib import Path
import os
file_path = Path("./People.pdf")
if not file_path.exists():
    from knwler.api import fetch_url
    url = "https://knwler.com/pdfs/People.pdf"
    metadata, bytes = await fetch_url(url, no_cache=True)
    with open(file_path, "wb") as f:
        f.write(bytes)
schema = Schema(entity_types=["person", "country"], relation_types=["is_from"])
r = await extract(file_path, schema=schema, config=Config())
print([e["name"] for e in r.graph.entities if e["type"] == "person"])


## Consolidation

If you have various documents around a common domain you likely want to merge the graphs extracted from these documents. 

In [ ]:
from knwler.api import *
schema1 = Schema(entity_types=["person", "location"], relation_types=["is_from"])
file_path1 = Path("./People.pdf")
if not file_path1.exists():
    url = "https://knwler.com/pdfs/People.pdf"
    metadata, bytes = await fetch_url(url, no_cache=True)
    with open(file_path1, "wb") as f:
        f.write(bytes)

schema2 = Schema(entity_types=["city", "country"], relation_types=["located_in"])
file_path2 = Path("./Places.pdf")
if not file_path2.exists():
    url = "https://knwler.com/pdfs/Places.pdf"
    metadata, bytes = await fetch_url(url, no_cache=True)
    with open(file_path2, "wb") as f:
        f.write(bytes)
config = Config(extraction_model="gemma3:12b")
g1 = await extract(file_path1, schema=schema1, config=config)
g2 = await extract(file_path2, schema=schema2, config=config)

g = await consolidate_document_graphs([g1, g2], config=config, include_chunks=True)
print(json.dumps(asdict(g), indent=2))

## Extras

There are some extra methods in the API you might find useful.

Rephrasing chunks can be useful for reporting but you can also use it prior to extracting graph, this is usually called chunk compression.

In [ ]:
from knwler.api import *
chunks = [Chunk(id="chunk1", text="> The $ cat is on the table."), Chunk(id="chunk2", text="The >>dog is in the garden.")]
rephrased_chunks = await rephrase_chunks(chunks, config=Config(extraction_model="gemma3:12b"))
for original, rephrased in zip(chunks, rephrased_chunks):
    print(f"Original: {original.text}\nRephrased: {rephrased}\n")

The title of a document is extracted from the first few paragraphs:

In [ ]:
from knwler.api import *
text ="""
Lovelace's educational and social exploits brought her into contact with scientists such as Andrew Crosse, Charles Babbage, David Brewster, Charles Wheatstone and Michael Faraday, and the author Charles Dickens, contacts which she used to further her education. Lovelace described her approach as "poetical science" and herself as an "Analyst (& Metaphysician)". When she was eighteen, Lovelace's mathematical talents led her to a long working relationship and friendship with fellow British mathematician Charles Babbage. She was particularly interested in Babbage's work on the analytical engine. Lovelace first met him on 5 June 1833, when she and her mother attended one of Charles Babbage's Saturday night soirées with their mutual friend, and Lovelace's private tutor, Mary Somerville.
"""
title = await extract_title(text, config=Config(extraction_model="gemma3:12b"))
print(f"Extracted title: {title}")

The summary of text or chunks is also handy for reporting:

In [ ]:
from knwler.api import *
file_path = Path("./HumanRights.pdf") 
if not file_path.exists():
    url = "https://knwler.com/pdfs/HumanRights.pdf"
    metadata, bytes = await fetch_url(url, no_cache=True)
    with open(file_path, "wb") as f:
        f.write(bytes)
content = await parse_pdf(file_path)
summary = await extract_summary(content, config=Config(extraction_model="gemma3:12b"))
print(f"Extracted summary: {summary}")        

## Export

With the API methods above you can turn the resulting output to a NetworkX graph, HTML and many other formats.

Let's first recreate a simple graph:

In [1]:
from knwler.api import *
from pathlib import Path
import os
file_path = Path("./People.pdf")
if not file_path.exists():
    from knwler.api import fetch_url
    url = "https://knwler.com/pdfs/People.pdf"
    metadata, bytes = await fetch_url(url, no_cache=True)
    with open(file_path, "wb") as f:
        f.write(bytes)
schema = Schema(entity_types=["person", "country"], relation_types=["is_from"])
doc = await extract(file_path, schema=schema, config=Config())


Output()

Turning this into a NetworkX graph is as simple as:

In [2]:
g = await create_network(doc)
print(f"Graph has {len(g.nodes)} nodes and {len(g.edges)} edges.")

Graph has 17 nodes and 8 edges.


You can use this NetworkX graph in all sorts of ways, notably to apply graph analytics (page rank and centrality in general).
Clustering in Knwler goes a step beyond standard graph analytics since it adds cluster labels to help understand what a cluster is mainly about:

In [6]:
from knwler.clustering import cluster_graph
g = await cluster_graph(doc.graph)
for cluster in g.clusters:
    print(f"Cluster {cluster['id']} with topics {cluster['topics']} contains nodes: {cluster['members']}")

Detected 8 clusters

Cluster 0 with topics ['International Research Collaboration'] contains nodes: ['Elena Petrova::person', 'Samuel Okoye::person']
Cluster 1 with topics ['Biodiversity Data Integration', 'Field Ecology'] contains nodes: ['Elena Petrova, Samuel Okoye::person', 'Lucía Fernández::person']
Cluster 2 with topics ['Data Engineering', 'International Collaboration'] contains nodes: ['Elena Petrova, Samuel Okoye, Lucía Fernández::person', 'Noah Williams::person']
Cluster 3 with topics ['Ethical Implications', 'Predictive Modeling'] contains nodes: ['Elena Petrova, Samuel Okoye, Lucía Fernández, Noah Williams::person', 'Sofia Dimitriou::person']
Cluster 4 with topics ['Computational Sustainability', 'International Research'] contains nodes: ['Elena Petrova, Samuel Okoye, Lucía Fernández, Noah Williams, Sofia Dimitriou, Hassan Al-Masri::person', 'Mei-Ling Chen::person']
Cluster 5 with topics ['Cross-Regional Data Analysis', 'Sustainability'] contains nodes: ['Arjun Mehta::person', 'Mei-Ling Chen, E

Finally, if you want to export a document graph to HTML you can assemble things into a dictionary and hand it over to the renderer.

In [10]:
html = await render_html(asdict(doc))

In [11]:
with open("output.html", "w") as f:
    f.write(html)